In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
from rdflib import Graph, Namespace, RDF, Literal

g = Graph()
EX = Namespace("http://example.org/minecraft#")

g.bind("ex", EX)


In [25]:
counters = {}

In [26]:
def count_name(name):
    if name not in counters:
        counters[name] = 1
    else:
        counters[name] += 1
    return counters[name]

### Текстовая модальность

In [27]:
text_file = "text1.tsv"
with open(text_file, encoding="utf-8") as f:
    for i, line in enumerate(f):
        
        if line.startswith("#") or not line.strip():
            continue
        
        parts = line.strip().split("\t")
        
        if len(parts) < 6:
            continue
        word = parts[2]
        category = parts[7]
        subcategory = parts[6]
        if category == '_' or subcategory == '_':
            continue
        
        print(f"Word: {word}; Category: {category}; Subcategory: {subcategory}")

        ann = EX[f"TextDescription_{i}"]
        g.add((ann, RDF.type, EX.TextAnnotation))
        g.add((ann, EX.text, Literal(word)))

        key = str.lower(subcategory)
        object = EX[f"{key.capitalize()}_{count_name(key)}"]
        g.add((object, RDF.type, EX[key]))
        g.add((ann, EX.hasDescription, object))

Word: вода; Category: BLOCK; Subcategory: WATER
Word: водой; Category: BLOCK; Subcategory: WATER
Word: выемку; Category: EVENT; Subcategory: ACTION
Word: воду; Category: BLOCK; Subcategory: WATER
Word: разносить; Category: EVENT; Subcategory: ACTION
Word: бега; Category: EVENT; Subcategory: RUN
Word: еду; Category: ITEM; Subcategory: FOOD
Word: вёдрами; Category: ITEM; Subcategory: TOOL
Word: железа; Category: ITEM; Subcategory: RESOURCE
Word: воды; Category: BLOCK; Subcategory: WATER
Word: убрать; Category: EVENT; Subcategory: ACTION
Word: грядки; Category: LOCATION; Subcategory: GARDEN
Word: собираем; Category: EVENT; Subcategory: ACTION
Word: семян; Category: ITEM; Subcategory: SEED
Word: огорода; Category: LOCATION; Subcategory: GARDEN


### Аудио модальность

In [28]:
from textgrids import TextGrid

audio_filepath = 'audio1.mp3'
tg = TextGrid("audio1.TextGrid")

tier = tg["sounds"]

for interval in tier:
    print(interval.text, interval.xmin, interval.xmax)

 0.0 40.847126580599365
HIT_CHICKEN 40.847126580599365 41.37927555613583
 41.37927555613583 41.60537714412297
CHICKEN 41.60537714412297 42.56378830924732
 42.56378830924732 48.26374302461519
SHEEP 48.26374302461519 49.06007388732584
 49.06007388732584 50.23017661382915
HIT_SHEEP 50.23017661382915 50.95237879491436
 50.95237879491436 51.34021071628824
ZOMBIE 51.34021071628824 51.85606941706339
 51.85606941706339 51.861499508650496
HIT 51.861499508650496 51.97553143197974
 51.97553143197974 51.9782464777733
SHEEP 51.9782464777733 52.621712330845455
 52.621712330845455 53.66037768367867
HIT_SHEEP 53.66037768367867 54.7192455431645
 54.7192455431645 54.9174191812121
HIT_SHEEP 54.9174191812121 56.50572097044084
 56.50572097044084 56.643781031248714
KILL_SHEEP 56.643781031248714 57.602192196373075
 57.602192196373075 62.123434467557196
SHEEP 62.123434467557196 62.601282527222594
 62.601282527222594 133.97897916666668


In [29]:
sound_tier = tg["sounds"]
for i, interval in enumerate(sound_tier):
    label = interval.text.strip()
    if not label:
        continue

    sound_event = EX[f"Sound_{i}"]
    g.add((sound_event, RDF.type, EX.sound))
    g.add((sound_event, EX.timeStart, Literal(interval.xmin)))
    g.add((sound_event, EX.timeEnd, Literal(interval.xmax)))
    g.add((sound_event, EX.filepath, Literal(audio_filepath)))
    
    label = str.lower(label)
    label_parts = label.split('_')
    if len(label_parts) == 1:
        name = label
    else:
        action_name = label_parts[0]
        name = label_parts[1]

    class_uri = EX[name]
    object = EX[f"{name.capitalize()}_{count_name(name)}"]
    g.add((object, RDF.type, class_uri))

    if len(label_parts) == 2:
        action_uri = EX[action_name]
        action_event = EX[f"{label.capitalize()}_{count_name(label)}"]
        g.add((action_event, RDF.type, action_uri))
        g.add((action_event, EX.actionTarget, object))
        g.add((action_event, EX.hasSound, sound_event))
    else:
        g.add((object, EX.hasSound, sound_event))

In [30]:
emotion_tier = tg["emotion"]
player = EX[f"Player_{1}"]
g.add((player, RDF.type, EX.player))
for i, interval in enumerate(emotion_tier):
    label = interval.text.strip()
    if not label:
        continue
    emotion_event = EX[f"{label.capitalize()}_{count_name(label)}"]
    g.add((sound_event, RDF.type, EX.emotion))
    g.add((sound_event, EX.timeStart, Literal(interval.xmin)))
    g.add((sound_event, EX.timeEnd, Literal(interval.xmax)))
    g.add((sound_event, EX.filepath, Literal(audio_filepath)))
    g.add((player, EX.hasEmotion, emotion_event))

### Видео модальность

In [31]:
import json

video_data = json.load(open("video.json"))

for i, (frame, content) in enumerate(video_data.items()):
    filename = content['filename']
    for j, region in enumerate(content["regions"]):
        
        ann = EX[f"View_{i}_{j}"]
        g.add((ann, RDF.type, EX.view))
        
        name = region["region_attributes"]["name"]
        class_uri = EX[name]
        object = EX[f"{name.capitalize()}_{count_name(name)}"]

        g.add((object, RDF.type, class_uri))
        g.add((object, EX.hasView, ann))
        g.add((ann, EX.filepath, Literal(filename)))
        
        shape = region["shape_attributes"]
        g.add((ann, EX.coordinates, Literal(str(shape))))

### SPARQL

In [32]:
q = """
PREFIX ex: <http://example.org/minecraft#>

SELECT ?ann
WHERE {
  ?ann ex:hasSound ?entity .
}
"""

for row in g.query(q):
  print(row)
  uri = str(row.ann)
  name = uri.split("#")[-1]
  print(name)

(rdflib.term.URIRef('http://example.org/minecraft#Hit_chicken_1'),)
Hit_chicken_1
(rdflib.term.URIRef('http://example.org/minecraft#Chicken_2'),)
Chicken_2
(rdflib.term.URIRef('http://example.org/minecraft#Sheep_1'),)
Sheep_1
(rdflib.term.URIRef('http://example.org/minecraft#Hit_sheep_1'),)
Hit_sheep_1
(rdflib.term.URIRef('http://example.org/minecraft#Zombie_1'),)
Zombie_1
(rdflib.term.URIRef('http://example.org/minecraft#Hit_1'),)
Hit_1
(rdflib.term.URIRef('http://example.org/minecraft#Sheep_3'),)
Sheep_3
(rdflib.term.URIRef('http://example.org/minecraft#Hit_sheep_2'),)
Hit_sheep_2
(rdflib.term.URIRef('http://example.org/minecraft#Hit_sheep_3'),)
Hit_sheep_3
(rdflib.term.URIRef('http://example.org/minecraft#Kill_sheep_1'),)
Kill_sheep_1
(rdflib.term.URIRef('http://example.org/minecraft#Sheep_7'),)
Sheep_7


In [33]:
q = """
PREFIX ex: <http://example.org/minecraft#>

SELECT ?entity
WHERE {
  ?ann ex:hasEmotion ?entity .
}
"""

for row in g.query(q):
  print(row)
  uri = str(row.entity)
  name = uri.split("#")[-1]
  print(name)

(rdflib.term.URIRef('http://example.org/minecraft#Neutral_1'),)
Neutral_1
(rdflib.term.URIRef('http://example.org/minecraft#Neutral_2'),)
Neutral_2
(rdflib.term.URIRef('http://example.org/minecraft#Neutral_3'),)
Neutral_3
(rdflib.term.URIRef('http://example.org/minecraft#Joking_1'),)
Joking_1
(rdflib.term.URIRef('http://example.org/minecraft#Neutral_4'),)
Neutral_4
(rdflib.term.URIRef('http://example.org/minecraft#Confusion_1'),)
Confusion_1
(rdflib.term.URIRef('http://example.org/minecraft#Neutral_5'),)
Neutral_5
(rdflib.term.URIRef('http://example.org/minecraft#Confusion_2'),)
Confusion_2
(rdflib.term.URIRef('http://example.org/minecraft#Neutral_6'),)
Neutral_6
(rdflib.term.URIRef('http://example.org/minecraft#Surprise_1'),)
Surprise_1


In [34]:
q = """
PREFIX ex: <http://example.org/minecraft#>

SELECT ?entity
WHERE {
  ?ann ex:hasDescription ?entity .
}
"""

for row in g.query(q):
  print(row)
  uri = str(row.entity)
  name = uri.split("#")[-1]
  print(name)

(rdflib.term.URIRef('http://example.org/minecraft#Water_1'),)
Water_1
(rdflib.term.URIRef('http://example.org/minecraft#Water_2'),)
Water_2
(rdflib.term.URIRef('http://example.org/minecraft#Action_1'),)
Action_1
(rdflib.term.URIRef('http://example.org/minecraft#Water_3'),)
Water_3
(rdflib.term.URIRef('http://example.org/minecraft#Action_2'),)
Action_2
(rdflib.term.URIRef('http://example.org/minecraft#Run_1'),)
Run_1
(rdflib.term.URIRef('http://example.org/minecraft#Food_1'),)
Food_1
(rdflib.term.URIRef('http://example.org/minecraft#Tool_1'),)
Tool_1
(rdflib.term.URIRef('http://example.org/minecraft#Resource_1'),)
Resource_1
(rdflib.term.URIRef('http://example.org/minecraft#Water_4'),)
Water_4
(rdflib.term.URIRef('http://example.org/minecraft#Action_3'),)
Action_3
(rdflib.term.URIRef('http://example.org/minecraft#Garden_1'),)
Garden_1
(rdflib.term.URIRef('http://example.org/minecraft#Action_4'),)
Action_4
(rdflib.term.URIRef('http://example.org/minecraft#Seed_1'),)
Seed_1
(rdflib.term.U

In [42]:
q = """
PREFIX ex: <http://example.org/minecraft#>

SELECT ?ann
WHERE {
  ?ann ex:hasView ?entity .
}
"""

for row in g.query(q):
  print(row)
  uri = str(row.ann)
  name = uri.split("#")[-1]
  print(name)

(rdflib.term.URIRef('http://example.org/minecraft#Cat_1'),)
Cat_1
(rdflib.term.URIRef('http://example.org/minecraft#Villager_1'),)
Villager_1
(rdflib.term.URIRef('http://example.org/minecraft#Composter_1'),)
Composter_1
(rdflib.term.URIRef('http://example.org/minecraft#Bed_1'),)
Bed_1
(rdflib.term.URIRef('http://example.org/minecraft#Torch_1'),)
Torch_1
(rdflib.term.URIRef('http://example.org/minecraft#Garden_3'),)
Garden_3
(rdflib.term.URIRef('http://example.org/minecraft#Ripe_wheat_1'),)
Ripe_wheat_1
(rdflib.term.URIRef('http://example.org/minecraft#Cactus_1'),)
Cactus_1
(rdflib.term.URIRef('http://example.org/minecraft#Villager_2'),)
Villager_2
(rdflib.term.URIRef('http://example.org/minecraft#Shield_1'),)
Shield_1
(rdflib.term.URIRef('http://example.org/minecraft#Health_1'),)
Health_1
(rdflib.term.URIRef('http://example.org/minecraft#Hunger_1'),)
Hunger_1
(rdflib.term.URIRef('http://example.org/minecraft#Shield_2'),)
Shield_2
(rdflib.term.URIRef('http://example.org/minecraft#Stone_

### Визуализация

In [38]:
import cosmograph as cg

In [39]:
def draw_beautiful_graph(edges, clusters = None, repulsion=1.5, spring=0.5, link_dist = 4, cluster = 0.05):
  nodes = list(set(node for edge in edges for node in edge[:2]))

  points = pd.DataFrame({
      "id": range(len(nodes)),
      "label": nodes,
  })

  if clusters != None:
    points["category"] = [clusters[node] for node in nodes]

  node_id_map = {name: idx for idx, name in enumerate(nodes)}

  links = pd.DataFrame({
      "source": [node_id_map[edge[0]] for edge in edges],
      "target": [node_id_map[edge[1]] for edge in edges],
  })

  if len(edges[0]) == 3:
    links["value"] = [edge[2] for edge in edges]

  widget = cg.cosmo(
      points=points,
      links=links,
      point_id_by="id",
      link_source_by="source",
      link_target_by="target",
      # point_color_by="category",
      link_include_columns=["value"],
      point_include_columns=["value"],
      point_label_by="label",
      simulation_repulsion=repulsion,
      simulation_link_spring=spring,
      simulation_link_distance=link_dist,
      simulation_cluster=cluster
  )

  return widget

In [40]:
from collections import defaultdict
def rdf_to_edges(g):
    edges = []

    for s, p, o in g:
        s = shorten(s)
        o = shorten(o)
        p = shorten(p)
        edges.append((s, o, p))
    return edges

def shorten(uri):
    uri = str(uri)
    return uri.split("/")[-1].split("#")[-1]

def extract_clusters(g):
    clusters = {}

    for s, p, o in g:
        if "type" in str(p):
            clusters[shorten(s)] = shorten(o)

    return clusters

In [41]:
edges = rdf_to_edges(g)
# clusters = extract_clusters(g)

draw_beautiful_graph(
    edges,
    # clusters=clusters,
    repulsion=2.0,
    spring=0.3,
    link_dist=6,
    cluster=0.1
)